# BlackBox — Phase 1: Memory Layers & Storage (real `mem0ai`)

This version replaces our hand-rolled store with the actual **`mem0ai`** open-source library.
The pieces:

- **Vector store**: Chroma (local, no server) — mem0 manages it internally now
- **Embedder**: HuggingFace `sentence-transformers/all-MiniLM-L6-v2` — same model you specified
- **LLM**: your lab's OpenAI-compatible router (`gpt-oss-20b`) — mem0 uses this to *extract*
  facts from raw conversation text when you call `.add()`

That last point is the big difference from our first version: real mem0 doesn't just embed
and store whatever text you give it — it uses the LLM to read the conversation and decide
*what's actually worth remembering*, then stores those extracted facts.

Running example: the Boeing 737 hydraulic systems engineer, same as before.


## 1. Install dependencies

In [1]:
!pip install mem0ai chromadb sentence-transformers python-dotenv google-adk litellm openai -q --break-system-packages

## 2. Save the lab's LLM config module

This is your team's existing helper for pointing at the lab's hosted vLLM endpoint.
We save it as `lab_llm_config.py` so we can import it like any other module.


## 3. Load the lab environment

`load_lab_env()` reads your `.env` file (or falls back to the lab defaults) and sets three
environment variables we'll reuse for mem0's LLM config: `LLM_API_BASE`, `LLM_MODEL_NAME`,
and `OPENAI_API_KEY`.


In [17]:
from lab_llm_config import load_lab_env
import os

# This sets LLM_API_BASE / LLM_MODEL_NAME / OPENAI_API_KEY as environment variables
load_lab_env()

print("LLM endpoint:", os.environ["LLM_API_BASE"])
print("LLM model:", os.environ["LLM_MODEL_NAME"])

LLM endpoint: http://10.0.10.51:8000/v1
LLM model: openai/gpt-oss-20b


## 4. Configure mem0

mem0 wants three things: where to store vectors (`vector_store`), how to turn text into
vectors (`embedder`), and which LLM does the fact-extraction (`llm`).

For the `llm` block, mem0's `"openai"` provider just needs a model name, a base URL, and an
API key — it talks to your lab router directly using the OpenAI SDK format, since your router
is already OpenAI-compatible. No LiteLLM wrapping needed here.


In [19]:
from mem0 import MemoryClient

config = {
    # Where the vectors + facts actually live on disk
    "vector_store": {
        "provider": "chroma",
        "config": {
            "collection_name": "blackbox_demo",  # like a table name
            "path": "./chroma_db",               # local folder, created automatically
        },
    },
    # Turns text into vectors, using the exact model you specified
    "embedder": {
        "provider": "huggingface",
        "config": {
            "model": "sentence-transformers/all-MiniLM-L6-v2",
        },
    },
    # Reads conversation text and decides what facts are worth remembering
    "llm": {
        "provider": "openai",
        "config": {
            "model": os.environ["LLM_MODEL_NAME"],       # e.g. "openai/gpt-oss-20b"
            "openai_base_url": os.environ["LLM_API_BASE"],  # your lab router
            "api_key": os.environ.get("OPENAI_API_KEY", "not-needed"),
            "temperature": 0.2,
            "max_tokens": 1024,
        },
    },
}

# This single call wires up the vector store + embedder + LLM together
#memory = MemoryClient.from_config(config)
print("mem0 Memory instance ready.")
memory = MemoryClient(api_key=os.environ.get("MEM0_API_KEY"))
print("mem0 Cloud MemoryClient ready.")


mem0 Memory instance ready.
mem0 Cloud MemoryClient ready.


## 5. Try it: the aircraft engineer, session by session

Unlike our first version, we're no longer hand-writing extracted facts. We give mem0 the
*raw conversation turn*, and the LLM in our config does the extraction for us.


### Session 1 — asks about hydraulic pressure specs

In [20]:
result = memory.add(
    "I work on Boeing 737 hydraulic systems. What's the standard hydraulic pressure spec?",
    user_id="eng_01",
)

print(result)   # shows exactly what facts mem0 decided to extract and store

{'event_id': '9e0a74a0-6dce-47ac-ad43-005383bcf055', 'status': 'PENDING'}


### Session 2 — mentions "our fleet" (mem0 should infer they manage multiple aircraft)

In [21]:
result = memory.add(
    "For our fleet, how often should we be checking hydraulic fluid levels?",
    user_id="eng_01",
)

print(result)

{'event_id': 'ce03ec01-10f1-498b-b191-6e16e9b7c1ed', 'status': 'PENDING'}


### Session 3 — asks about troubleshooting, not just theory

In [22]:
result = memory.add(
    "The pressure gauge is reading low during preflight checks. Walk me through troubleshooting steps, not just the specs.",
    user_id="eng_01",
)

print(result)

{'event_id': 'f7d2670b-bd8b-48e3-bfd8-3861d9d5401b', 'status': 'PENDING'}


## 6. See everything mem0 has stored for this user

Useful for sanity-checking what actually got extracted across all three sessions.


In [24]:
# Pass user_id inside the filters dictionary
all_memories = memory.get_all(filters={"user_id": "eng_01"})

# Print the extracted memories
for item in all_memories.get("results", []):
    print("•", item["memory"])


• User observed a low hydraulic pressure gauge reading during preflight checks on a Boeing 737 and asked for troubleshooting steps
• User is seeking guidance on how often to check hydraulic fluid levels for their fleet of Boeing 737 aircraft
• User works on Boeing 737 hydraulic systems, handling their maintenance and engineering aspects


## 7. The payoff: Session 4

A new session starts. Short question, zero restated context.


In [26]:
results = memory.search(
    "what's the backup system?", filters={"user_id": "eng_01"}
)

# Print formatted search results
for item in results.get("results", []):
    print("•", item["memory"])
    print("  Score:", item.get("score"))


• User works on Boeing 737 hydraulic systems, handling their maintenance and engineering aspects
  Score: 0.1601
• User observed a low hydraulic pressure gauge reading during preflight checks on a Boeing 737 and asked for troubleshooting steps
  Score: 0.12
• User is seeking guidance on how often to check hydraulic fluid levels for their fleet of Boeing 737 aircraft
  Score: 0.108


In [27]:
# Once you've seen the shape above, pull just the memory text out of it
for r in results["results"]:
    print("-", r["memory"])

- User works on Boeing 737 hydraulic systems, handling their maintenance and engineering aspects
- User observed a low hydraulic pressure gauge reading during preflight checks on a Boeing 737 and asked for troubleshooting steps
- User is seeking guidance on how often to check hydraulic fluid levels for their fleet of Boeing 737 aircraft


If Session 4 pulled back the Boeing 737 / fleet / troubleshooting facts without you
repeating any of it, the full mem0 loop is working: **raw conversation in → LLM extracts
facts → embedder turns them into vectors → Chroma stores them → search retrieves by
meaning later.**

**Next up — Phase 2:** looking at what mem0 does with *conflicting* facts (e.g. the user later
says they've moved to the 787 fleet) and how importance/decay show up in `get_all()` over time.
